# M1a — TSP do Beerlink Distribuição (versão SOLUÇÃO)

**Treinamento de Otimização Aplicada — Genoa para Gradus**

Encontrar a **ordem ótima de visita** de 1 caminhão grande percorrendo o CD em Pinheiros + 8 bares em SP. Capacidade ignorada (M1b adiciona).

**Variáveis:** $x_{ij} \in \{0,1\}$ se o caminhão vai de $i$ a $j$, $u_i \ge 0$ para eliminar subtour (MTZ).

**FO:** $\min \sum_{i,j} \text{dist}_{ij} \, x_{ij}$

**Restrições:**
- Cada nó tem 1 arco de saída: $\sum_j x_{ij} = 1\ \forall i$
- Cada nó tem 1 arco de entrada: $\sum_i x_{ij} = 1\ \forall j$
- MTZ (elimina subtours): $u_i - u_j + n \cdot x_{ij} \le n - 1\ \forall i, j \ne 0,\ i \ne j$

## Setup

In [ ]:
%pip install -q ortools gurobipy

In [ ]:
import math, time
from itertools import product

# Coordenadas aproximadas (lat, lon) - CD + 8 bares em SP
COORDS = {
    'CD':            (-23.567, -46.685),
    'Centro':        (-23.553, -46.635),
    'Pinheiros':     (-23.565, -46.685),
    'Vila Madalena': (-23.555, -46.692),
    'Moema':         (-23.605, -46.665),
    'Tatuape':       (-23.539, -46.572),
    'Lapa':          (-23.521, -46.706),
    'Itaim':         (-23.583, -46.671),
    'Brooklin':      (-23.612, -46.690),
}
NODES = list(COORDS.keys())
N = range(len(NODES))   # 0 = CD, 1..8 = bares
n = len(NODES)

# Matriz de distancias (km) - aproximacao Euclidiana lat-lon * 111
def km(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) * 111

dist = [[km(COORDS[NODES[i]], COORDS[NODES[j]]) for j in N] for i in N]

print(f'{n} nos ({n-1} bares + CD)')
print(f'Distancia max entre nos: {max(dist[i][j] for i, j in product(N, N)):.1f} km')
print(f'Bar mais distante do CD: {NODES[max(range(1,n), key=lambda i: dist[0][i])]}')

## 1) OR-Tools (pywraplp, CBC)

In [ ]:
from ortools.linear_solver import pywraplp

def solve_tsp_ortools():
    s = pywraplp.Solver.CreateSolver('CBC')

    # Variaveis
    x = {(i, j): s.IntVar(0, 1, f'x[{i},{j}]')
         for i, j in product(N, N) if i != j}
    u = [s.NumVar(0, n, f'u[{i}]') for i in N]

    # (1) cada no tem 1 saida
    for i in N:
        s.Add(sum(x[i, j] for j in N if j != i) == 1)

    # (2) cada no tem 1 entrada
    for j in N:
        s.Add(sum(x[i, j] for i in N if i != j) == 1)

    # (3) MTZ: elimina subtours
    for i, j in product(N, N):
        if i != j and i != 0 and j != 0:
            s.Add(u[i] - u[j] + n * x[i, j] <= n - 1)

    # FO
    s.Minimize(sum(dist[i][j] * x[i, j]
                   for i, j in product(N, N) if i != j))

    t0 = time.time()
    s.Solve()
    elapsed = time.time() - t0

    # Extrai rota
    route = [0]
    node = 0
    for _ in range(n):
        for j in N:
            if j != node and x[node, j].solution_value() > 0.5:
                route.append(j)
                node = j
                break
    return {'rota': route, 'dist_total': s.Objective().Value(), 'tempo_ms': int(elapsed * 1000)}

res_or = solve_tsp_ortools()
print(f'OR-Tools: dist total = {res_or["dist_total"]:.1f} km, tempo = {res_or["tempo_ms"]} ms')
print(f'Rota: {" -> ".join(NODES[i] for i in res_or["rota"])}')

## 2) Gurobi (gurobipy)

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_tsp_gurobi():
    m = gp.Model('tsp')
    m.Params.OutputFlag = 0

    # Variaveis (tupledict)
    x = m.addVars([(i, j) for i, j in product(N, N) if i != j],
                  vtype=GRB.BINARY, name='x')
    u = m.addVars(N, lb=0, ub=n, name='u')

    # (1) saida
    m.addConstrs((x.sum(i, '*') == 1 for i in N), name='out')
    # (2) entrada
    m.addConstrs((x.sum('*', j) == 1 for j in N), name='in')
    # (3) MTZ
    m.addConstrs((u[i] - u[j] + n * x[i, j] <= n - 1
                  for i, j in product(N, N)
                  if i != j and i != 0 and j != 0),
                 name='mtz')

    # FO
    m.setObjective(gp.quicksum(dist[i][j] * x[i, j]
                               for i, j in product(N, N) if i != j),
                   GRB.MINIMIZE)

    t0 = time.time()
    m.optimize()
    elapsed = time.time() - t0

    # Extrai rota
    route = [0]
    node = 0
    for _ in range(n):
        for j in N:
            if j != node and x[node, j].X > 0.5:
                route.append(j)
                node = j
                break
    return {'rota': route, 'dist_total': m.ObjVal, 'tempo_ms': int(elapsed * 1000)}

res_gb = solve_tsp_gurobi()
print(f'Gurobi: dist total = {res_gb["dist_total"]:.1f} km, tempo = {res_gb["tempo_ms"]} ms')
print(f'Rota: {" -> ".join(NODES[i] for i in res_gb["rota"])}')

## 3) Comparação e visualização

In [ ]:
import pandas as pd

comp = pd.DataFrame([
    {'Solver': 'OR-Tools (CBC)',   'Distância (km)': f'{res_or["dist_total"]:.1f}', 'Tempo (ms)': res_or['tempo_ms']},
    {'Solver': 'Gurobi (gurobipy)', 'Distância (km)': f'{res_gb["dist_total"]:.1f}', 'Tempo (ms)': res_gb['tempo_ms']},
])
comp

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 7))
for i, n_name in enumerate(NODES):
    lat, lon = COORDS[n_name]
    cor = '#f29200' if i == 0 else '#1f4e79'
    sz  = 280 if i == 0 else 120
    ax.scatter(lon, lat, s=sz, c=cor, zorder=3, edgecolors='white', linewidth=2)
    ax.annotate(n_name, (lon, lat), xytext=(6, 6), textcoords='offset points', fontsize=10)

# Desenha a rota Gurobi
rota = res_gb['rota']
for t in range(len(rota) - 1):
    i, j = rota[t], rota[t+1]
    lat_i, lon_i = COORDS[NODES[i]]
    lat_j, lon_j = COORDS[NODES[j]]
    ax.annotate('', xy=(lon_j, lat_j), xytext=(lon_i, lat_i),
                arrowprops=dict(arrowstyle='->', color='#2e7d32', lw=2))

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'TSP Beerlink — rota ótima ({res_gb["dist_total"]:.1f} km)')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Lições do M1a

- **MTZ é necessário** em TSP: sem ele, o solver pode formar subtours (ciclos desconectados do depósito) — válidos matematicamente mas inviáveis operacionalmente.
- **Os 2 solvers acham o mesmo ótimo** (TSP de 9 nós é trivial). A diferença começa a importar quando n > 30.
- **Capacidade fica fora do M1a** — é o que motiva o M1b: 340 cx de demanda total NÃO cabem em 1 caminhão de 100 cx. O CVRP é a próxima etapa.